# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an end-to-end workflow for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s as defined in the Croissant metadata.

In [ ]:
# Get the list of available record sets
record_sets = []
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        record_sets.append(rs['@id'] if isinstance(rs, dict) else rs)
elif hasattr(metadata, 'recordSet'):
    for rs in metadata.recordSet:
        record_sets.append(rs['@id'] if isinstance(rs, dict) else rs)
else:
    # Try to extract from direct metadata (for datasets with a single record set)
    if hasattr(metadata, 'recordSet'):
        record_sets.append(metadata.recordSet['@id'])

# If record sets are not in metadata, attempt to extract via dataset API
if not record_sets:
    try:
        record_sets = dataset.record_sets
    except AttributeError:
        print("No record sets found in Croissant schema.")

if not record_sets:
    raise ValueError('No record sets found in the dataset metadata.')

print("Available record sets (by @id):")
for idx, rsid in enumerate(record_sets, 1):
    print(f"{idx}: {rsid}")

# Display field and column IDs for each record set
for rsid in record_sets:
    print(f"\n--- Fields and columns in RecordSet @id = {rsid} ---")
    try:
        rs_obj = dataset.metadata.record_sets[[r['@id'] for r in dataset.metadata.record_sets].index(rsid)]
    except Exception as e:
        print(f"Could not find detailed definition for RecordSet {rsid} in metadata.")
        continue
    if 'fields' in rs_obj:
        for field in rs_obj['fields']:
            fid = field['@id'] if isinstance(field, dict) else field
            print(f"Field: {fid}")
    if 'columns' in rs_obj:
        for col in rs_obj['columns']:
            cid = col['@id'] if isinstance(col, dict) else col
            print(f"\tColumn: {cid}")
    # For simple schemas
    if hasattr(rs_obj, 'fields'):
        for field in rs_obj.fields:
            fid = field['@id'] if isinstance(field, dict) else field
            print(f"Field: {fid}")

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set using their @id
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for RecordSet {record_set_id} ...")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) == 0:
            print(f"No records found in RecordSet {record_set_id}.")
            continue
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id} with columns: {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Could not extract records for {record_set_id} (error: {e})")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming fields, and grouping data for further analysis.

> **Note:** You must choose a numeric field and a group field from the previously extracted columns. Replace `<numeric_field_id>` and `<group_field_id>` with an actual column name as printed above.

In [ ]:
# Example: EDA on the first record set found
first_rs = record_sets[0]
df = dataframes[first_rs]

# Pick a numeric field and a group field by inspecting df.columns (replace below if needed)
# Typical column names might be 'cr:coverage', 'cr:peptide_count', etc., depending on schema
# Print to help the user select
print("Columns in DataFrame:", df.columns.tolist())

# Example guesses for common omics/protein mass spec datasets
possible_numeric_fields = [c for c in df.columns if any(k in c.lower() for k in ['coverage', 'count', 'mw', 'pi', 'abundance'])]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    numeric_field_id = df.columns[0]  # fallback
print(f"Selected numeric field: {numeric_field_id}")

# Try by default to use a sample or experimental group as group_field
possible_group_fields = [c for c in df.columns if any(k in c.lower() for k in ['sample', 'experiment', 'group', 'replicate', 'condition'])]
group_field = possible_group_fields[0] if possible_group_fields else df.columns[0]
print(f"Selected group field: {group_field}")

# Explore numeric distribution
pd.options.mode.chained_assignment = None  # Suppress SettingWithCopy warning
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {filtered_df.shape[0]} records")
display(filtered_df.head())

# Normalize the numeric field
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print(f"Field {numeric_field_id} is not numeric and cannot be normalized.")

# Group and summarize
if group_field in filtered_df.columns and pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} grouped by {group_field}:")
    display(grouped_df.head())
else:
    print(f"Grouping not performed (group field missing or numeric field non-numeric).")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check if numeric analysis was possible
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group if group field is categorical
    if pd.api.types.is_categorical_dtype(df[group_field]) or df[group_field].dtype == object:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print(f"Cannot plot {numeric_field_id} because it is not numeric.")

## 6. Conclusion
This notebook demonstrated how to:
- Load a FAIR² Croissant dataset using `mlcroissant`, referencing all entities by `@id`.
- Inspect available record sets, fields, and columns with their `@id`s.
- Extract data as DataFrames for analysis.
- Perform basic EDA including filtering, normalization, and grouping by `@id`.
- Visualize numeric data fields across groups.

You can now extend this workflow for more advanced analyses or link with other Croissant datasets.